# Selecting interesting test dwarfs


The goal of this notebook is to identify a small subset of dwarf galaxies 
to validate catalogue analysis on.

we want to select dwarf galaxies with
- High and low S/N
- extreme and weak emission
- old and young
- metal poor and rich
- close and far
- massive and low mass
- of all morphologies

To acchieve this, we select dwarf galaxies with properties near manually placed points in different plots as to pick out interesting features in different dimensions. In general, we use STARLIGHT derived stellar properties 

In [ ]:
from astropy.table import Table, hstack
import numpy as np
import arya
import matplotlib.pyplot as plt

In [ ]:
from scipy.ndimage import median_filter

In [ ]:
import tomllib

In [ ]:
import os
from pathlib import Path

In [ ]:
outfile = Path("../analysis/selected_targetid_new.txt")

In [ ]:
if outfile.exists():
    os.remove(outfile)

In [ ]:
arya.use("mnras")

In [ ]:
# dwarf_cat = Table.read("../data/dwarfs_good.fits")

In [ ]:
dwarf_main = Table.read("../data/dwarfs_good.fits", hdu=1)
dwarf_z = Table.read("../data/dwarfs_good.fits", hdu=2)
dwarf_phot = Table.read("../data/dwarfs_good.fits", hdu=3)
dwarf_fsf = Table.read("../data/dwarfs_good.fits", hdu=4)

In [ ]:
dwarf_mel = Table.read("../data/stellar_mass_emline_dwarfs.fits")

In [ ]:
QUANTILES = [0.001, 0.02, 0.16, 0.5, 0.84, 0.98, 0.999]

In [ ]:
exclude = ~np.isin(dwarf_main["TARGETID"], dwarf_mel["targetid"])

In [ ]:
np.mean(exclude)

In [ ]:
# exclude = [484] # outside of LSS09 range

In [ ]:
def select_by_quantile(x, q=QUANTILES):

    qs = np.quantile(x[np.isfinite(x)], q)
    idxs = []
    for qv in qs:
        delta = np.abs(x - qv)
        delta[~np.isfinite(delta)] = np.inf
        delta[exclude] = np.inf
        idxs.append(int(np.argmin(delta)))

    return idxs

In [ ]:
x_bpt = np.log10(dwarf_fsf["NII_6584_FLUX"] / dwarf_fsf["HALPHA_FLUX"])  - 0.15
y_bpt = np.log10(dwarf_fsf["OIII_5007_FLUX"] / dwarf_fsf["HBETA_FLUX"]) - 1.4
r_bpt = x_bpt + y_bpt
# r_bpt_err = np.sqrt(
#     (dwarf_fsf["nii_6583_flux_ivar"]**0.5 / dwarf_cat["nii_6583_flux"])**2 + 
#     (dwarf_fsf["halpha_flux_ivar"]**0.5 / dwarf_cat["halpha_flux"])**2 + 
#     (dwarf_fsf["oiii_5007_flux_ivar"]**0.5 / dwarf_cat["oiii5007_flux"])**2 + 
#     (dwarf_fsf["hbeta_flux_ivar"]**0.5 / dwarf_cat["hbeta_flux"])**2
# ) / np.log(10)

In [ ]:
plt.hist(r_bpt[np.isfinite(r_bpt)], bins=np.linspace(-2, 2, 100))

In [ ]:
bins = np.linspace(15, 28, 100)
plt.hist(dwarf_phot["MAG_R"], bins)


In [ ]:
strat_vars = [
    ("SNR", dwarf_fsf["SNR_R"], 0*dwarf_fsf["SNR_R"]),
    ("mass", np.log10(dwarf_fsf["LOG_MSTAR_FASTSPEC"]), 1/np.log(10) * dwarf_fsf["LOG_MSTAR_FASTSPEC_IVAR"]**-0.5 / dwarf_fsf["LOG_MSTAR_FASTSPEC"]), # dwarf catalogue mass, not the SMELmass
    # ("age", dwarf_fsf[""], dwarf_cat["ageerr_cg"]),
    ("rmag", dwarf_phot["MAG_R"], dwarf_phot["MAG_R_ERR"]),
    # ("$Z$", dwarf_fsf["MGD"], dwarf_fsf["zerr_cg"]),
    ("emission", r_bpt, r_bpt * 0),
    ("g-r", dwarf_phot["MAG_G"] - dwarf_phot["MAG_R"], np.sqrt(dwarf_phot["MAG_G_ERR"]**2 + dwarf_phot["MAG_R_ERR"]**2)),
    ("redshift", dwarf_main["Z"], 0*dwarf_main["Z"]),
]

    

In [ ]:
def select_highest(x):
    all_idxs = np.arange(len(x))
    
    filt = ~exclude
    
    perm = np.argsort(x[filt])[::-1]
    idxs = all_idxs[filt][perm]
    
    return idxs[:len(QUANTILES)]

In [ ]:
dwarf_idxs = []

for (name, x, xerr) in strat_vars:
    if name == "SNR":
        idxs = select_highest(x)
    else:
        idxs = select_by_quantile(x)
    dwarf_idxs = [*dwarf_idxs, *idxs]

len(dwarf_idxs)

In [ ]:
for i in range(len(strat_vars)):
    name, x, xerr = strat_vars[i]
    q = np.linspace(0, 1, len(x))
    perm = np.argsort(x)

    plt.scatter(q, x[perm], color="k", lw=0, s=2)
    plt.plot(q, median_filter(x[perm] + xerr[perm], size=5), color="C0", alpha=0.1, lw=0.3)
    plt.plot(q, median_filter(x[perm] - xerr[perm], size=5), color="C0", alpha=0.1, lw=0.3)

    invperm = np.empty_like(perm)
    invperm[perm] = np.arange(perm.size)

    qs = [np.mean(x < x[ii]) for ii in dwarf_idxs]
    plt.errorbar(qs, x[dwarf_idxs], yerr=xerr[dwarf_idxs], fmt="o", capsize=0)
    nq = len(QUANTILES)

    this_slice = slice(i*nq, (i+1)*nq)
    print(i, this_slice)
    plt.errorbar(qs[this_slice], x[dwarf_idxs][this_slice], yerr=xerr[dwarf_idxs][this_slice], fmt="o", zorder=5, capsize=0)
    plt.ylabel(name)
    plt.xlabel("quantile")
    plt.show()

In [ ]:
target_ids = dwarf_main["TARGETID"][dwarf_idxs]


In [ ]:
with open("../analysis/selected_targetid.txt", "w") as f:
    f.write("targetid = [\n")

    for target_id in target_ids:
         f.write(f"    {target_id},\n")
    
    f.write("]")

In [ ]:
assert all(np.isin(target_ids, dwarf_mel["targetid"]))

In [ ]:
print(list([int(i) for i in dwarf_main["TARGETID"][dwarf_idxs].value]))

In [ ]:
with open("../analysis/selected_targetid.txt", "rb") as f:
    h = tomllib.load(f)
    print(h["targetid"])

# Plots

In [ ]:
dwarf_cat = hstack([dwarf_main, dwarf_fsf, dwarf_phot])

In [ ]:
dwarf_cat["LOG

In [ ]:
fig, ax = plt.subplots()

x = dwarf_cat["MAG_G_1"] - dwarf_cat["MAG_R_1"]
y = dwarf_cat["LOG_MSTAR_M24"]
plt.scatter(x, y, s=0.3, lw=0, alpha=0.01, color="k")


plt.scatter(x[dwarf_idxs], y[dwarf_idxs], color="C1")
ax.set(
    xlim = (-0.5, 1.5),
    ylim = (6, 9.1),
    xlabel = "g - r [mag]",
    ylabel = r"$\log M_\star / M_\odot$ [CIGALE]",
)

plt.savefig("figures/dwarf_selected_Mstar_gr.png")

In [ ]:
fig, ax = plt.subplots()
ax.set(
    xlabel = r"log([NII] $\lambda$6583 / H$\alpha$)",
    ylabel = r"log([OIII] $\lambda$5007 / H$\beta$)",
    xlim = (-2, 1),
    ylim = (-1, 2),
)


x = np.log10(dwarf_cat["nii6583_flux"] / dwarf_cat["halpha_flux"])
y = np.log10(dwarf_cat["oiii5007_flux"] / dwarf_cat["hbeta_flux"])

plt.scatter(x, y, s=0.3, alpha=0.3, c="k", lw=0, clim=(0, 1))
plt.scatter(x[dwarf_idxs], y[dwarf_idxs], color="C1")

# x_model = np.linspace(-2, 0.15, 1000)
# y_model = 0.61 / (x_model - 0.15) + 1.4 # Eq. 1 from Zou+2024, +0.1 dex modelling uncertainty shift in both directions
# plt.plot(x_model, y_model)

x_model = np.linspace(-2, -0.05, 1000)
y_model = 0.61 / (x_model - 0.05) + 1.3
plt.plot(x_model, y_model)
plt.savefig("figures/dwarf_selected_BPT.png")

In [ ]:
fig, ax = plt.subplots()
ax.set(
    xlabel = r"log($M_\star / M_\odot$)",
    ylabel = r"log(SFR / M$_\odot$ yr$^{-1}$)",
    xlim = (6, 9.6),
    ylim = (-3, 2),
)

x = np.log10(dwarf_cat["mass_cg"])
y = np.log10(dwarf_cat["sfr_cg"] )

plt.scatter(x, y, s=0.1, color="black", alpha=0.03, lw=0)
plt.scatter(x[dwarf_idxs], y[dwarf_idxs], color="C1")
plt.savefig("figures/dwarf_selected_SFR_vs_Mstar.png")

In [ ]:
fig, ax = plt.subplots()
ax.set(
    xlabel = r"log($M_\star / M_\odot$)",
    ylabel = r"$Z$ (starlight)",
    xlim = (6, 9.6),
)

x = np.log10(dwarf_cat["mass_cg"])
y = dwarf_cat["z_cg"] 

plt.scatter(x, y, s=0.1, color="black", alpha=0.03, lw=0)
plt.scatter(x[dwarf_idxs], y[dwarf_idxs], color="C1")
plt.savefig("figures/dwarf_selected_SFR_vs_Mstar.png")

# LSS image retrieval

In [ ]:
import sys
sys.path.append("..")

In [ ]:
import desi_utils

In [ ]:
from desi_utils.get_data import download_ls_image, get_spectra
from desi_utils.plotting import _get_img_data, color_clip_image, show_colour_image

In [ ]:
def ra_dec_axis():
    plt.xlabel(r"Right Ascension")
    plt.ylabel(r"Declination")

In [ ]:
zeropoint = [-0.00, 0.000, 0.00]

In [ ]:
imgs = []

for i, idx in enumerate(dwarf_idxs):    
    print("loading image", i)
    img = download_ls_image(dwarf_main["RA"][idx], dwarf_main["DEC"][idx])

    imgs.append(img)

In [ ]:
bad_imgs = np.where([np.sum(_get_img_data(img) == 0.) > 65_000 for img in imgs])[0]
bad_imgs

In [ ]:
n_cols = len(QUANTILES)
n_rows = len(dwarf_idxs) // n_cols 

print(n_rows * n_cols)
fig, axs = plt.subplots(n_rows, n_cols, figsize=(2*n_cols, 2*n_rows), sharex=True, sharey=True)

for i, idx in enumerate(dwarf_idxs):
    row = i // n_cols
    col = i % n_cols
    ax = axs[row, col]
    plt.sca(ax)
    
    img = imgs[i]
    img_data = _get_img_data(img)
    
    show_colour_image(color_clip_image(img_data + zeropoint, scale=1e-2, vmax=1e1), 
                      clim=(0, 1), figsize=(3, 3), subplot_kw=dict(projection=img.wcs[0]), ax=ax, fig=fig)
    # plt.scatter(img_data.shape[0] / 2, img_data.shape[1] / 2)
    plt.title(dwarf_main["TARGETID"][idx])

    if col == 0:
        plt.ylabel(strat_vars[row][0])


plt.tight_layout()
plt.savefig("figures/dwarf_selected_cutouts.png")

In [ ]:
from scipy.ndimage import gaussian_filter1d

In [ ]:
n_cols = len(QUANTILES)
n_rows = len(dwarf_idxs) // n_cols 


fig, axs = plt.subplots(2*n_rows, n_cols, figsize=(2*n_cols, 2*2*n_rows), sharex="row",
   gridspec_kw = dict(
       wspace=0.05, hspace=0.05, height_ratios = [1, 3] * n_rows)
   )


spec = get_spectra(dwarf_main["TARGETID"][dwarf_idxs])

for i, idx in enumerate(dwarf_idxs):
    row = 2*(i // n_cols)
    col = i % n_cols
    ax = axs[row, col]
    plt.sca(ax)


    y = gaussian_filter1d(spec.flux.value[i, :], 10)
    plt.plot(spec.spectral_axis.value, y)
    ax.set_axis_off()
    ax.set(
        ylim=(0, np.max(y) * 1.05)
    )
    plt.title(dwarf_main["TARGETID"][idx])

            
    



    row += 1
    ax = axs[row, col]
    plt.sca(ax)
    img = imgs[i]
    img_data = _get_img_data(img)
    
    show_colour_image(color_clip_image(img_data + zeropoint, scale=1e-2, vmax=1e1), 
                      clim=(0, 1), figsize=(3, 3), subplot_kw=dict(projection=img.wcs[0]), ax=ax, fig=fig)
    # plt.scatter(img_data.shape[0] / 2, img_data.shape[1] / 2)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    name = strat_vars[row // 2][0]
    val = strat_vars[row // 2][1][idx]
    plt.title(f"{name} = {val:.3g}")

    if col == 0:
        plt.ylabel(strat_vars[row // 2][0])


plt.savefig("figures/dwarf_selected_cutouts_spectra.png")